# PyTorch CUDA/ROCm/MPS or CPU training

Create a workspace and save its key first. Run cells in order. Credentials use hidden prompts.


In [ ]:
%pip install -q "crashlens[jupyter,gpu] @ git+https://github.com/kavinsaravan/ML-Failure-Doctor.git#subdirectory=crashlens-sdk"

In [ ]:
import os, getpass, requests
from crashlens import WorkloadTracker
api_url = (os.getenv("CRASHLENS_URL") or input("Backend HTTPS URL: ")).strip().rstrip("/")
api_key = os.getenv("CRASHLENS_API_KEY") or getpass.getpass("Your saved CrashLens key: ")
fireworks_key = os.getenv("FIREWORKS_API_KEY") or getpass.getpass("Your Fireworks key (blank for rules): ")
fireworks_model = (os.getenv("FIREWORKS_MODEL") or input("Tool-capable Fireworks model: ")) if fireworks_key else None
tracker = WorkloadTracker(api_url, api_key=api_key, fireworks_api_key=fireworks_key or None, fireworks_model=fireworks_model)
response = requests.get(api_url + "/session", headers=tracker.headers, timeout=15)
response.raise_for_status()
print("Connected to your private workspace")

In [ ]:
import torch, time
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Training device:", device)
synchronize = torch.cuda.synchronize if device == "cuda" else (torch.mps.synchronize if device == "mps" else lambda: None)
with tracker.track("Portable PyTorch training") as success_id:
    model = torch.nn.Linear(128, 32).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=.01)
    inputs = torch.randn(16, 128, device=device)
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline:
        optimizer.zero_grad()
        loss = model(inputs).square().mean()
        loss.backward(); optimizer.step(); synchronize()
        time.sleep(.02)
    print("Training complete")
print("Workload ID:", success_id)

In [ ]:
try:
    with tracker.track("Real missing dependency") as dependency_id:
        import crashlens_intentionally_missing_dependency_847263
except ModuleNotFoundError:
    print("Expected import failure recorded; workload", dependency_id)

In [ ]:
# This may consume your Fireworks credits if credentials were supplied.
report = tracker.diagnose(dependency_id)
print("Source:", report["source"])
print("Root cause:", report["root_cause"])
print("Recommended fix:", report["recommended_fix"])

In [ ]:
response = requests.get(api_url + "/workloads", headers=tracker.headers, timeout=15)
response.raise_for_status()
for workload in response.json():
    print(workload["id"], workload["name"], workload["status"])
print("Open your frontend dashboard using the same CrashLens key.")